# Waste2Cash — Modèle de Prédiction d'Écoulement des Invendus

Ce notebook construit et entraîne l'IA de Waste2Cash. Deux briques :

1. **Modèle d'écoulement (par lot)** : prédit, pour un lot donné (catégorie, DLC, distance, etc.),
   - la probabilité qu'il soit effectivement écoulé (`was_sold`),
   - le temps estimé avant écoulement (`time_to_sell_hours`).
2. **Modèle de prévision de volume (par catégorie, horizon 6 mois)** : prévoit le volume d'invendus
   attendu par catégorie de produit pour piloter les partenariats et la capacité de collecte à l'avance.

> Environnement offline : nous utilisons `scikit-learn` (RandomForest / GradientBoosting) plutôt que
> XGBoost/Prophet. En production, ces modèles peuvent être remplacés par XGBoost et Prophet pour de
> meilleures performances — l'architecture et le pipeline de features restent identiques.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import (accuracy_score, roc_auc_score, classification_report,
                              mean_absolute_error, r2_score)

pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (9, 5)

## 1. Chargement des données

In [ ]:
df = pd.read_csv("waste2cash_dataset.csv", parse_dates=["collection_datetime"])
print(df.shape)
df.head()

In [ ]:
df.info()
df.isna().sum()

## 2. Feature engineering

On dérive quelques variables temporelles supplémentaires et on prépare les colonnes catégorielles / numériques.

In [ ]:
df["month_of_year"] = df["collection_datetime"].dt.month
df["day_of_week"] = df["collection_datetime"].dt.day_name()
df["hour_of_day"] = df["collection_datetime"].dt.hour
df["is_weekend"] = df["day_of_week"].isin(["Saturday", "Sunday"]).astype(int)

# Ratio DLC restante vs temps déjà écoulé - proxy d'urgence
df["urgency_ratio"] = df["days_until_expiry"] / (df["days_until_expiry"].mean() + 1)

CATEGORICAL_FEATURES = [
    "product_category", "product_subcategory", "storage_condition", "packaging_type",
    "pickup_city", "day_of_week", "season",
]
NUMERIC_FEATURES = [
    "quantity_kg", "unit_price_initial", "days_until_expiry", "is_holiday",
    "distance_to_partner_km", "partner_storage_capacity", "partner_typical_demand",
    "weather_temperature", "rainfall_mm", "month_of_year", "hour_of_day",
    "is_weekend", "urgency_ratio",
]

FEATURES = CATEGORICAL_FEATURES + NUMERIC_FEATURES
df[FEATURES].head()

## 3. Modèle A — Probabilité d'écoulement (classification)

Cible : `was_sold` (1 = le lot a été repris par un partenaire, 0 = perte / invendu).

In [ ]:
X = df[FEATURES]
y_clf = df["was_sold"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

preprocessor = ColumnTransformer(transformers=[
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
    ("num", StandardScaler(), NUMERIC_FEATURES),
])

clf_pipeline = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300, max_depth=12, min_samples_leaf=5,
        class_weight="balanced", random_state=42, n_jobs=-1
    )),
])

clf_pipeline.fit(X_train, y_train)

In [ ]:
y_pred = clf_pipeline.predict(X_test)
y_proba = clf_pipeline.predict_proba(X_test)[:, 1]

print("Accuracy :", round(accuracy_score(y_test, y_pred), 3))
print("ROC AUC  :", round(roc_auc_score(y_test, y_proba), 3))
print()
print(classification_report(y_test, y_pred, target_names=["Non écoulé", "Écoulé"]))

## 4. Modèle B — Temps estimé avant écoulement (régression)

Cible : `time_to_sell_hours`. Utile pour prioriser les lots urgents dans le matching.

In [ ]:
y_reg = df["time_to_sell_hours"]

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)

reg_pipeline = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", GradientBoostingRegressor(
        n_estimators=300, max_depth=4, learning_rate=0.05, random_state=42
    )),
])

reg_pipeline.fit(X_train_r, y_train_r)

pred_r = reg_pipeline.predict(X_test_r)
print("MAE (heures) :", round(mean_absolute_error(y_test_r, pred_r), 2))
print("R²           :", round(r2_score(y_test_r, pred_r), 3))

In [ ]:
plt.scatter(y_test_r, pred_r, alpha=0.2, s=10)
plt.plot([0, y_test_r.max()], [0, y_test_r.max()], color="red", linestyle="--")
plt.xlabel("Temps réel d'écoulement (h)")
plt.ylabel("Temps prédit (h)")
plt.title("Modèle B — Réel vs Prédit")
plt.show()

## 5. Importance des variables

Comprendre quels paramètres pèsent le plus dans la décision d'écoulement — utile pour affiner
la collecte de données et pour expliquer le modèle aux partenaires/investisseurs.

In [ ]:
importances = clf_pipeline.named_steps["model"].feature_importances_
feature_names = clf_pipeline.named_steps["prep"].get_feature_names_out()

imp_df = pd.DataFrame({"feature": feature_names, "importance": importances})
imp_df = imp_df.sort_values("importance", ascending=False).head(15)

plt.barh(imp_df["feature"][::-1], imp_df["importance"][::-1], color="#1f5fbf")
plt.title("Top 15 variables les plus influentes — Modèle d'écoulement")
plt.tight_layout()
plt.show()

## 6. Prévision de volume d'invendus par catégorie — horizon 6 mois

Objectif business : anticiper, catégorie par catégorie, le **volume d'invendus attendu sur les
6 prochains mois** afin de dimensionner les partenariats (Socofrais, Dabali Express, etc.) et la
capacité de collecte à l'avance.

Approche : agrégation mensuelle du volume (`quantity_kg`) par catégorie, puis régression sur les
composantes temporelles (mois, tendance, saison) — équivalent simplifié d'un modèle Prophet.

In [ ]:
monthly = (
    df.groupby([pd.Grouper(key="collection_datetime", freq="MS"), "product_category"])["quantity_kg"]
    .sum()
    .reset_index()
    .rename(columns={"collection_datetime": "month", "quantity_kg": "total_kg"})
)
monthly["month_index"] = (monthly["month"].dt.year - monthly["month"].dt.year.min()) * 12 + monthly["month"].dt.month
monthly["month_num"] = monthly["month"].dt.month
monthly.head()

In [ ]:
def forecast_category(cat_df, horizon_months=6):
    """Régression simple : tendance linéaire + saisonnalité mensuelle (dummies)."""
    cat_df = cat_df.sort_values("month_index").copy()
    X = pd.get_dummies(cat_df["month_num"], prefix="m")
    X["trend"] = cat_df["month_index"]
    y = cat_df["total_kg"]

    model = RandomForestRegressor(n_estimators=200, max_depth=4, random_state=42)
    model.fit(X, y)

    last_index = cat_df["month_index"].max()
    last_month_num = cat_df.loc[cat_df["month_index"] == last_index, "month_num"].iloc[0]

    future_rows = []
    for h in range(1, horizon_months + 1):
        fut_index = last_index + h
        fut_month_num = ((last_month_num - 1 + h) % 12) + 1
        future_rows.append({"month_index": fut_index, "month_num": fut_month_num})

    future_df = pd.DataFrame(future_rows)
    X_future = pd.get_dummies(future_df["month_num"], prefix="m")
    X_future["trend"] = future_df["month_index"]
    X_future = X_future.reindex(columns=X.columns, fill_value=0)

    future_df["predicted_kg"] = model.predict(X_future)
    return future_df

forecasts = {}
for cat in df["product_category"].unique():
    cat_df = monthly[monthly["product_category"] == cat]
    forecasts[cat] = forecast_category(cat_df, horizon_months=6)

for cat, f in forecasts.items():
    print(f"\n{cat} — prévision des 6 prochains mois (kg) :")
    print(f[["month_index", "predicted_kg"]].round(1).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
colors = plt.cm.tab10.colors
for i, (cat, f) in enumerate(forecasts.items()):
    hist = monthly[monthly["product_category"] == cat]
    ax.plot(hist["month_index"], hist["total_kg"], label=f"{cat} (historique)", color=colors[i % 10])
    ax.plot(f["month_index"], f["predicted_kg"], linestyle="--", color=colors[i % 10])

ax.set_xlabel("Index mensuel")
ax.set_ylabel("Volume total (kg)")
ax.set_title("Historique + prévision à 6 mois, par catégorie")
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.show()

## 7. Priorisation du stock à écouler (aide au matching)

Fonction simple combinant :
- la probabilité d'écoulement prédite par le Modèle A,
- l'urgence (DLC restante),

pour produire un **score de priorité** par lot — utilisé par la plateforme pour ordonner les
lots à proposer en premier aux partenaires.

In [ ]:
def compute_priority_score(row_df):
    proba_sold = clf_pipeline.predict_proba(row_df[FEATURES])[:, 1]
    urgency = 1 / (row_df["days_until_expiry"] + 1)
    # Score : on priorise les lots urgents ET à forte probabilité d'être repris rapidement
    score = 0.6 * urgency + 0.4 * proba_sold
    return score

sample = df.sample(10, random_state=1).copy()
sample["priority_score"] = compute_priority_score(sample)
sample[["product_category", "days_until_expiry", "priority_score"]].sort_values(
    "priority_score", ascending=False
)

## 8. Export des modèles pour intégration à la plateforme

On sauvegarde les pipelines entraînés avec `joblib`. Le backend de la plateforme (ex: FastAPI)
pourra les charger directement pour scorer un nouveau lot en temps réel.

In [ ]:
joblib.dump(clf_pipeline, "model_probabilite_ecoulement.joblib")
joblib.dump(reg_pipeline, "model_temps_ecoulement.joblib")
print("Modèles sauvegardés : model_probabilite_ecoulement.joblib, model_temps_ecoulement.joblib")

## 9. Exemple d'inférence (utilisation en production)

Ci-dessous, la fonction telle qu'elle serait appelée par l'API de la plateforme lorsqu'un nouveau
lot est enregistré après collecte.

In [ ]:
def predict_new_lot(lot: dict) -> dict:
    """
    lot : dictionnaire avec les mêmes clés que FEATURES
    Retourne la probabilité d'écoulement, le temps estimé, et un score de priorité.
    """
    row = pd.DataFrame([lot])[FEATURES]
    proba = clf_pipeline.predict_proba(row)[0, 1]
    temps_estime = reg_pipeline.predict(row)[0]
    urgency = 1 / (lot["days_until_expiry"] + 1)
    priority = 0.6 * urgency + 0.4 * proba
    return {
        "probabilite_ecoulement": round(float(proba), 3),
        "temps_estime_heures": round(float(temps_estime), 1),
        "score_priorite": round(float(priority), 3),
    }

exemple_lot = {
    "product_category": "Fruits & Légumes", "product_subcategory": "Agrumes",
    "storage_condition": "Frais", "packaging_type": "Carton", "pickup_city": "Abidjan",
    "day_of_week": "Friday", "season": "Pluies",
    "quantity_kg": 80, "unit_price_initial": 1.3, "days_until_expiry": 2, "is_holiday": 0,
    "distance_to_partner_km": 8.5, "partner_storage_capacity": 500, "partner_typical_demand": 300,
    "weather_temperature": 29.5, "rainfall_mm": 3.2, "month_of_year": 7, "hour_of_day": 14,
    "is_weekend": 0, "urgency_ratio": 0.5,
}

predict_new_lot(exemple_lot)

## Résumé

- **Modèle A (classification)** : prédit la probabilité qu'un lot soit effectivement écoulé — sert à
  alerter tôt sur les lots à risque et à décider d'une remise plus agressive.
- **Modèle B (régression)** : estime le temps d'écoulement — utile pour le SLA de matching.
- **Prévision par catégorie (6 mois)** : pilotage stratégique — anticiper les volumes pour dimensionner
  les partenariats commerciaux et les tournées de collecte.
- **Score de priorité** : fonction combinée directement branchable sur l'algorithme de matching de
  la plateforme.

**Prochaine étape recommandée** : dès que des données réelles remplacent le dataset synthétique
(après quelques mois de pilote), ré-entraîner ces mêmes pipelines — l'architecture ne change pas.